## Document Loading & Chunk Splitting

In [ ]:
from rag.utils.pdf import CustomPDFLoader

In [ ]:
loader = CustomPDFLoader("<DOC_URL>")
transformed_docs = loader.load(chunk_docs=True)

In [ ]:
# Examine chunk lengths
[len(i.page_content) for i in transformed_docs]

In [ ]:
len(transformed_docs)

## Using LCEL-based chains and Langfuse callback

In [ ]:
from rag.chains import rag_chain_with_source

In [ ]:
rag_chain_with_source.get_graph().draw_png("chain_dag.png")

In [ ]:
from rag.utils.callbacks import langfuse_handler_from_config

# Check connection to Langfuse host
langfuse_handler = langfuse_handler_from_config()
langfuse_handler.auth_check()

In [ ]:
hp_installer_bot_chain.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
result = rag_chain_with_source.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})
# result is a dictionary with the keys: context, query, answer
# the sources used can be post processed
# the chatbot system prompt can also be modified
result

In [ ]:
rag_chain_with_source.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
config1 = rag_chain_with_source.with_config(configurable={"temperature": 0.3})
config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
config1.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})

In [ ]:
config2 = rag_chain_with_source.with_config(configurable={"model_name": "gpt-4-turbo","temperature": 0.3})
config2.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
config2.invoke("What information do you have to hand? Which installations guides do you have?", config={"callbacks": [langfuse_handler]})

In [ ]:
config3 = rag_chain_with_source.with_config(configurable={"model_name": "gpt-4-turbo","temperature": 0.3})

In [ ]:
from rag.vector_databases.retrievers import chatbot_retriever

In [ ]:
# Documents can still be retrieved this way, but embeddings generation of the query is not logged.
chatbot_retriever.invoke("What is your name?", config={"callbacks": [langfuse_handler]})

In [ ]:
chatbot_retriever.get_graph().draw_png("retriever_dag.png")

In [ ]:
retriever_config1 = chatbot_retriever.with_config(configurable={
        "search_kwargs": {"score_threshold": 0.74},
        "search_type": "similarity_score_threshold",
    }
)
res = retriever_config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
len(res) # no documents have similarity > 0.74

In [ ]:
retriever_config1 = chatbot_retriever.with_config(configurable={"search_kwargs": {"k": 7}})
res = retriever_config1.invoke("What is your name?", config={"callbacks": [langfuse_handler]})
len(res) # returns k=7 documents

# Testing RAG Chain served in REST API via Langserve RemoteRunnable

In [ ]:
from langserve import RemoteRunnable

question = "What information do you have to hand? Which installations guides do you have?"

api_runnable = RemoteRunnable("<API_URL>/")
api_runnable.invoke(question, config={"configurable": {"model_name": "gpt-3.5-turbo"}})

In [ ]:
questions = [
    "What is your name?",
    "Who are you?",
    "What information do you have?"
]

questions = questions * 50

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# TODO: Add some pauses between invokations to avoid rate limiting on LLM generation
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(api_runnable.invoke, question, config={"configurable": {"model_name": "gpt-3.5-turbo"}}) for question in questions]
    results = [future.result() for future in futures]

# Evaluation - Retrieving the trace data

Traces are results from invoking a chain, parsed into a nested structure of observervations from the chain DAG

In [ ]:
from langfuse import Langfuse

langfuse = Langfuse()

assert langfuse.auth_check()

### Demo of Langfuse Python API to retrieve traces

In [ ]:
traces = langfuse.client.trace.list(name="whatsapp_chatbot_chain") # defaults to returning one page, each page defaults to 50 traces
# traces.dict() # contains two keys: data and meta

In [ ]:
# Get all traces for a trace name
def get_traces(name=None, limit=10000, user_id=None):
    all_data = []
    page = 1
 
    while True:
        response = langfuse.client.trace.list(
            name=name, page=page, user_id=user_id, order_by=None
        )
        if not response.data:
            break
        page += 1
        all_data.extend(response.data)
        if len(all_data) > limit:
            break
 
    return all_data[:limit]

In [ ]:
all_traces = get_traces(name="whatsapp_chatbot_chain")

#### Extracting directly from `traces.data` excludes more fine-grained metadata

In [ ]:
traces.data[0].dict()

In [ ]:
# Still very useful!
data = [
    {
        "trace_id": trace.id,
        "user_id": trace.user_id,
        "session_date": trace.session_id,
        "input": trace.input,
        "output": trace.output,
        "metadata": trace.metadata,
        "total_cost": trace.total_cost,
        "latency": trace.latency,
        "tags": trace.tags,
        "version": trace.version
    }
    for trace in traces.data
]

The `trace.id`, which is `trace_id` from the above `data` variable can be used to retrieve more fine-grained generation metadata.

Feel free to explore what other metadata could be useful from other observations in the retrieved trace.

In [ ]:
test = langfuse.client.observations.get_many(trace_id=traces.data[0].dict()["id"])
test.data

In [ ]:
test_gen = [o for o in test.data if o.type.lower() == "generation"]
test_gen[0].dict()

Below, we combine data directly from `all_traces`, with data from generation observations within each trace.

In [ ]:
evaluation_batch = {
    "question": [],
    "context": [],
    "response": [],
    "trace_id": [],
    "generation_metadata": [],
}
 
for t in all_traces:
    # Get the observations for the trace
    trace_id = t.id
    observations = langfuse.client.observations.get_many(trace_id=trace_id)

    # More data from trace observations
    for o in observations.data:
        if o.type.lower() == "generation":
            metadata = dict(
                model = o.model,
                temperature = o.model_parameters["temperature"],
                input_tokens = o.usage.input,
                output_tokens = o.usage.output,
                total_tokens = o.usage.total,
                input_cost = o.calculated_input_cost,
                output_cost = o.calculated_output_cost,
                total_cost = o.calculated_total_cost,
            )
        # can also get data from `o.name == "VectorStoreRetriever"`` etc.

    # Data directly from traces, some traces generation failed therefore no output
    try:
        output = t.dict()["output"]
        evaluation_batch['question'].append(output["query"])
        evaluation_batch['context'].append(output["context"])
        evaluation_batch['response'].append(output["answer"])
        evaluation_batch['trace_id'].append(trace_id)
        evaluation_batch['generation_metadata'].append(metadata)
    except KeyError:
        print(f"{trace_id}: No output")

In [ ]:
data = [dict(zip(evaluation_batch, t)) for t in zip(*evaluation_batch.values())]

In [ ]:
import pandas as pd

df = pd.DataFrame(data)

In [ ]:
df["generation_metadata"][0]

In [ ]:
df